In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, recall_score

In [ ]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_val
)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

train_input = torch.tensor(X_train, dtype=torch.float32, device=device)
train_label = torch.tensor(y_train, dtype=torch.long, device=device)
val_input = torch.tensor(X_val, dtype=torch.float32, device=device)
val_label = torch.tensor(y_val, dtype=torch.long, device=device)
test_input = torch.tensor(X_test, dtype=torch.float32, device=device)
test_label = torch.tensor(y_test, dtype=torch.long, device=device)

In [ ]:
input_size = 30
hidden_1 = 10
hidden_2 = 5
output_size = 2
degree = 4

In [ ]:
c1 = nn.Parameter(torch.empty(input_size, hidden_1, degree + 1, device=device))
b1 = nn.Parameter(torch.zeros(hidden_1, device=device))

c2 = nn.Parameter(torch.empty(hidden_1, hidden_2, degree + 1, device=device))
b2 = nn.Parameter(torch.zeros(hidden_2, device=device))

c3 = nn.Parameter(torch.empty(hidden_2, output_size, degree + 1, device=device))
b3 = nn.Parameter(torch.zeros(output_size, device=device))

nn.init.xavier_normal_(c1)
nn.init.xavier_normal_(c2)
nn.init.xavier_normal_(c3)

In [ ]:
parameters = [c1, b1, c2, b2, c3, b3]

In [ ]:
def make_chebyshev_polynomials(x, degree):
    x = torch.tanh(x)

    polynomials = [torch.ones_like(x)]

    if degree >= 1:
        polynomials.append(x)

    for n in range(2, degree + 1):
        next_polynomial = 2 * x * polynomials[-1] - polynomials[-2]
        polynomials.append(next_polynomial)

    return torch.stack(polynomials, dim=2)

In [ ]:
def chebyshev_layer(x, coefficients, bias):
    layer_degree = coefficients.shape[2] - 1
    chebyshev_values = make_chebyshev_polynomials(x, layer_degree)
    output = torch.einsum('bid,iod->bo', chebyshev_values, coefficients)
    return output + bias

In [ ]:
def forward(x):
    x = chebyshev_layer(x, c1, b1)
    x = nn.functional.layer_norm(x, (hidden_1,))

    x = chebyshev_layer(x, c2, b2)
    x = nn.functional.layer_norm(x, (hidden_2,))

    x = chebyshev_layer(x, c3, b3)
    return x

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(parameters, lr=0.005, weight_decay=0.0004)
l1_strength = 1e-6

In [ ]:
max_epochs = 3000
patience = 150
best_val_loss = float('inf')
best_parameters = None
patience_counter = 0

for epoch in range(max_epochs):
    train_logits = forward(train_input)
    classification_loss = criterion(train_logits, train_label)
    l1_penalty = c1.abs().sum() + c2.abs().sum() + c3.abs().sum()
    loss = classification_loss + l1_strength * l1_penalty

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        val_logits = forward(val_input)
        val_loss = criterion(val_logits, val_label).item()

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_parameters = [p.detach().clone() for p in parameters]
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 100 == 0:
        print(
            f'Epoch {epoch:4d} | '
            f'train loss {classification_loss.item():.4f} | '
            f'validation loss {val_loss:.4f}'
        )

    if patience_counter >= patience:
        print('Early stopping at epoch', epoch)
        break

with torch.no_grad():
    for parameter, best_value in zip(parameters, best_parameters):
        parameter.copy_(best_value)


In [ ]:
with torch.no_grad():
    test_logits = forward(test_input)
    probabilities = torch.softmax(test_logits, dim=1)
    predictions = probabilities.argmax(dim=1)

y_true = test_label.cpu().numpy()
y_pred = predictions.cpu().numpy()

print(classification_report(y_true, y_pred, target_names=data.target_names))
print('Recall:', recall_score(y_true, y_pred, zero_division=0))


In [ ]:
with torch.no_grad():
    feature_strength = c1.abs().sum(dim=(1, 2)).cpu().numpy()

ranking = np.argsort(feature_strength)[::-1]
for index in ranking[:10]:
    print(f'{data.feature_names[index]:30s} {feature_strength[index]:.4f}')


In [ ]:
def print_edge_equation(coefficients, input_index, output_index):
    values = (
        coefficients[input_index, output_index]
        .detach()
        .cpu()
        .numpy()
    )

    equation = (
        f"{values[0]:+.4f} T0(z) "
        f"{values[1]:+.4f} T1(z) "
        f"{values[2]:+.4f} T2(z) "
        f"{values[3]:+.4f} T3(z) "
        f"{values[4]:+.4f} T4(z)"
    )

    print(equation)
    print("where z = tanh(x)")

In [ ]:
print_edge_equation(
    c1,
    input_index=0,
    output_index=0
)

In [ ]:
def print_expanded_edge(coefficients, input_index, output_index):
    c = (
        coefficients[input_index, output_index]
        .detach()
        .cpu()
    )

    constant = c[0] - c[2] + c[4]
    linear = c[1] - 3 * c[3]
    quadratic = 2 * c[2] - 8 * c[4]
    cubic = 4 * c[3]
    quartic = 8 * c[4]

    print(
        f"φ(z) = "
        f"{constant.item():+.4f} "
        f"{linear.item():+.4f}z "
        f"{quadratic.item():+.4f}z² "
        f"{cubic.item():+.4f}z³ "
        f"{quartic.item():+.4f}z⁴"
    )

    print("where z = tanh(x)")

In [ ]:
print_expanded_edge(
    c1,
    input_index=0,
    output_index=0
)